# RFSoC ↔ Prototype Compute Fabric HAL Playground

An interactive companion for the hardware-abstraction-layer prototype. This notebook loads the same source bundle as the local project, runs its checks and demos, and makes the dense matrix and feedback behavior visible.

> Educational prototype only: simulated instruments and a prototype compute fabric, not a real hardware integration.

## System view

`Experiment request → compiler-style validation → queued HAL → private driver adapters → RFSoC test system + prototype compute fabric → capture and evidence`

The notebook keeps the physical wiring private. It exercises logical ports, programmable weights, queued lifecycle operations, measurement evidence, access roles, and simulator-versus-driver-backed parity.

In [ ]:
# Upload the prepared analog-fabric-hal-playground.zip bundle from this project.
# In Colab: run this cell, choose the ZIP file, then continue.
from google.colab import files
from pathlib import Path
import os
import shutil
import zipfile

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload analog-fabric-hal-playground.zip')

project_dir = Path('/content/analog_fabric')
shutil.rmtree(project_dir, ignore_errors=True)
project_dir.mkdir()
with zipfile.ZipFile(zip_name) as bundle:
    bundle.extractall(project_dir)
os.chdir(project_dir)
print('Loaded:', project_dir)

In [ ]:
# Verify the HAL contract before running demonstrations.
!python3 -m unittest -q

## Demo 1 — queued experiment and evidence

This run shows logical source and capture jobs, per-item weight results, role admission, measurement verification, safe-stop, audit events, and simulator/driver-backed parity.

In [ ]:
!python3 demo.py

## Demo 3 — dense 4×4 matrix layer

All 16 input-to-output edge weights are applied. Rows are output nodes and columns are input nodes.

In [ ]:
import json
import matplotlib.pyplot as plt

with open('demo_full_matrix.json') as source:
    spec = json.load(source)
weights = spec['weights']
matrix = [[next(item['weight'] for item in weights if item['source'] == column and item['destination'] == row)
           for column in range(4)] for row in range(4)]

figure, axis = plt.subplots(figsize=(6, 5))
image = axis.imshow(matrix, cmap='coolwarm', vmin=-1, vmax=1)
axis.set(title='Demo 3: programmable 4×4 edge weights', xlabel='source input', ylabel='destination output')
axis.set_xticks(range(4), [f'in{index + 1}' for index in range(4)])
axis.set_yticks(range(4), [f'out{index + 1}' for index in range(4)])
for row in range(4):
    for column in range(4):
        axis.text(column, row, f'{matrix[row][column]:.2f}', ha='center', va='center')
figure.colorbar(image, label='programmed edge weight')
plt.show()
!python3 demo_full_matrix.py

## Demo 4 — 8×8 feedback fabric

Four logical inputs plus four logical outputs form an eight-node state. All 64 directed edges are programmed; the trace shows how one feedback update influences the next.

In [ ]:
!python3 demo_recurrent.py

## Nonideal backend comparison

The final demonstration applies deterministic gain and offset to the driver-backed path. A mismatch is expected: it models a calibration or regression signal rather than a queue failure.

In [ ]:
!python3 demo_nonideal.py